# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook demonstrates loading, exploring, and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is available
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'
# Load dataset
dataset = mlc.Dataset(croissant_url)
# Metadata object (do not subscript, just print attributes)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available **record sets**, **fields**, and their `@id`s.

We'll enumerate all record sets and their fields as defined by the Croissant schema. Each entity is referenced using its `@id`.

In [ ]:
# Get Croissant objects definitions
print("Record Sets (by @id) and their fields/columns:")
for record_set in dataset.record_sets:
    print(f"\nRecord Set: {record_set['@id']}")
    fields = record_set.get('field', [])
    if fields and not isinstance(fields, list):
        fields = [fields]
    print("  Fields (by @id):")
    for field in fields:
        field_id = field if isinstance(field, str) else field.get('@id', str(field))
        print(f"    - {field_id}")
    columns = record_set.get('column', [])
    if columns and not isinstance(columns, list):
        columns = [columns]
    if columns:
        print("  Columns (by @id):")
        for col in columns:
            col_id = col if isinstance(col, str) else col.get('@id', str(col))
            print(f"    - {col_id}")

## 3. Data Extraction

Load data from the main record set(s) into Pandas DataFrame(s) for analysis.

Use the record set and field `@id`s from the overview above.

Below, we select the primary survey record set (update the `main_record_set_id` with the main `@id` as printed above):

In [ ]:
# Usually, the main record set is something like 'cr:RecordSet_1', but list all record sets to pick the right ID.
# For this dataset, we can detect (or set manually) the main record set.

# --- Identify main record set ---
all_recordset_ids = [rs['@id'] for rs in dataset.record_sets]
print('All RecordSet @ids:', all_recordset_ids)
# You may need to adjust the next line based on the printed output above
main_record_set_id = all_recordset_ids[0] if all_recordset_ids else None
assert main_record_set_id, 'No record sets found!'

# If there are multiple, add them to the list below
record_sets = [main_record_set_id]
dataframes = {}

for record_set in record_sets:
    # Load records as list of dicts
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)

print(f"Loaded columns for '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())  # columns correspond to field/column @id

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Here we:
- Select a numeric field by its `@id` (adjust as needed)
- Filter records where the value is above a threshold
- Normalize that field
- Group and aggregate by a demographic field (`@id`)

Refer to column names (field `@id`s) printed in the previous cell.

In [ ]:
# Set field @ids based on actual DataFrame columns (you may need to check them)
# Example: 'cr:PHQ9_score', 'cr:age', 'cr:gender', etc. Replace with the appropriate @ids.
# If you are unsure, print(df.columns) as shown above.

df = dataframes[main_record_set_id]

# Choose a numeric field, e.g., PHQ-9 depression score
numeric_field = [col for col in df.columns if 'phq' in col.lower() and 'score' in col.lower()]
if numeric_field:
    numeric_field = numeric_field[0]
else:
    numeric_field = df.select_dtypes(include='number').columns[0]  # fallback

print('Selected numeric field @id:', numeric_field)
threshold = 10

# Filtering
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    filtered_df = df[df[numeric_field] > threshold].copy()
else:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    filtered_df = df[df[numeric_field] > threshold].copy()

print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping
group_field = [col for col in df.columns if 'gender' in col.lower() or 'sex' in col.lower()]
if group_field:
    group_field = group_field[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    display(grouped_df)
else:
    print('No suitable group field found for grouping.')

## 5. Visualization

Visualize the distribution of a numeric field (e.g., PHQ-9, GAD-7) and group differences.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic histogram
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# Boxplot by group if available
if group_field:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion

- The [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) was successfully loaded using `mlcroissant` by Croissant record set and field/column `@id`s.
- Key numeric mental health indicators (e.g., PHQ-9 score) were selected by Croissant field `@id` and used for filtering, normalization, and visualization.
- Data was grouped by demographic fields (e.g., gender), showing mean scores per group.

This notebook demonstrates how to use Croissant metadata to programmatically and reproducibly access FAIR datasets for further analysis.